# In-vivo multi-seed summary

Loads the per-seed result CSVs produced by `scripts/invivo_multi_seed.py`
(`results/invivo_multiseed_{cellina,cellina-gat,spatialprop}.csv`) and reports
mean ± std **over seeds** for each (model, type, baseline), analogous to the
single-run `summary_fmt` table in `pfish_analysis.ipynb` §4.3.

Note: the `mean` baseline is model-agnostic (no model calls at all) and is only
ever saved under `model == 'cellina'` by the training script, so it appears once
in this summary rather than once per model.

In [1]:
import glob
import os

import numpy as np
import pandas as pd

RESULTS_DIR = "../../results"
METRICS = ["Pearson", "Precision", "E-distance", "RMSE_LFC"]

## Load and combine

In [2]:
csv_paths = sorted(glob.glob(os.path.join(RESULTS_DIR, "invivo_multiseed_*.csv")))
print("found:\n " + "\n ".join(csv_paths))

df = pd.concat([pd.read_csv(p) for p in csv_paths], ignore_index=True)
print(f"\n{len(df)} rows | models: {sorted(df['baseline'].unique())}")
df.groupby("baseline")["seed"].nunique().rename("n_seeds").to_frame()

found:
 ../../results/invivo_multiseed_cellina-gat.csv
 ../../results/invivo_multiseed_cellina.csv
 ../../results/invivo_multiseed_spatialprop.csv
 ../../results/invivo_multiseed_terra.csv

810 rows | models: ['Cellina-GAT', 'Cellina-GAT-random', 'Cellina-base', 'Cellina-base-random', 'SpatialProp', 'SpatialProp-random', 'mean', 'terra-lora-ep5', 'terra-lora-ep5-random']


,n_seeds
baseline,
Cellina-GAT,5
Cellina-GAT-random,5
Cellina-base,5
Cellina-base-random,5
SpatialProp,5
SpatialProp-random,5
mean,5
terra-lora-ep5,5
terra-lora-ep5-random,5


In [3]:
df

,KO,type,baseline,Pearson,Precision,E-distance,RMSE_LFC,seed,model
0,MAP2K2,full,Cellina-GAT,0.890551,0.90,4.223041,0.528561,0,cellina-gat
1,MAP2K2,KO-specific,Cellina-GAT,0.933303,1.00,4.310908,0.439353,0,cellina-gat
2,IRAK1,full,Cellina-GAT,0.687919,1.00,4.834281,0.530005,0,cellina-gat
3,IRAK1,KO-specific,Cellina-GAT,0.853757,1.00,4.850016,0.307246,0,cellina-gat
4,IRF7,full,Cellina-GAT,0.820346,0.80,4.284441,0.320334,0,cellina-gat
...,...,...,...,...,...,...,...,...,...
805,NFKBIA,KO-specific,terra-lora-ep5-random,0.605995,0.80,1.521694,0.411428,4,terra
806,LBP,full,terra-lora-ep5-random,0.109561,0.70,1.602434,0.443281,4,terra
807,LBP,KO-specific,terra-lora-ep5-random,-0.407908,0.40,1.581340,0.434518,4,terra
808,MAP2K3,full,terra-lora-ep5-random,0.568972,0.72,1.481416,0.519746,4,terra


## Per-seed summary

Average each metric over the 9 KOs within a (model, seed, type, baseline) --
one score per training run, the same quantity `pfish_analysis.ipynb`'s single-run
`summary_fmt` reports (just not yet aggregated across seeds).

In [4]:
per_seed = df.groupby(["seed", "type", "baseline"])[METRICS].mean().reset_index()
per_seed

,seed,type,baseline,Pearson,Precision,E-distance,RMSE_LFC
0,0,KO-specific,Cellina-GAT,0.327000,0.655556,4.379881,0.375864
1,0,KO-specific,Cellina-GAT-random,-0.009515,0.482222,4.406111,0.431572
2,0,KO-specific,Cellina-base,0.311139,0.711111,3.938490,0.413531
3,0,KO-specific,Cellina-base-random,0.028063,0.488889,3.861616,0.427254
4,0,KO-specific,SpatialProp,0.193953,0.622222,7.462140,0.590956
...,...,...,...,...,...,...,...
85,4,full,SpatialProp,0.057080,0.544444,7.449240,0.741008
86,4,full,SpatialProp-random,0.001612,0.508889,7.449390,0.764045
87,4,full,mean,0.416440,0.633333,0.136340,0.555013
88,4,full,terra-lora-ep5,0.327223,0.633333,1.536851,0.518379


## Summary: mean ± std over seeds

In [5]:
# numeric mean/std, for programmatic use
summary = per_seed.groupby(["type", "baseline"])[METRICS].agg(["mean", "std"])
summary

Pearson           Precision            \
                                       mean       std      mean       std   
type        baseline                                                        
KO-specific Cellina-GAT            0.362779  0.175805  0.642222  0.054092   
            Cellina-GAT-random     0.083704  0.065131  0.515111  0.031087   
            Cellina-base           0.325105  0.084850  0.677778  0.030429   
            Cellina-base-random   -0.001643  0.052959  0.483111  0.035784   
            SpatialProp            0.157013  0.058304  0.553333  0.064502   
            SpatialProp-random     0.107952  0.049187  0.522222  0.060103   
            mean                   0.057844  0.062096  0.513333  0.034605   
            terra-lora-ep5         0.030877  0.073396  0.533333  0.047140   
            terra-lora-ep5-random -0.027116  0.055972  0.498667  0.037238   
full        Cellina-GAT            0.397697  0.223854  0.666667  0.119928   
            Cellina-GAT-random    -0.013420  0.171072  0.503556  0.121173   
            Cellina-base           0.532982  0.050007  0.760000  0.027889   
            Cellina-base-random    0.112281  0.234712  0.552000  0.055768   
            SpatialProp            0.260795  0.158920  0.613333  0.080277   
            SpatialProp-random     0.194597  0.165737  0.578667  0.094341   
            mean                   0.506392  0.127449  0.726667  0.077619   
            terra-lora-ep5         0.149604  0.115669  0.553333  0.052938   
            terra-lora-ep5-random  0.086937  0.136842  0.529778  0.059475   

                                  E-distance            RMSE_LFC            
                                        mean       std      mean       std  
type        baseline                                                        
KO-specific Cellina-GAT             3.998774  0.257699  0.420112  0.035188  
            Cellina-GAT-random      4.060631  0.234005  0.467230  0.032688  
            Cellina-base            4.160288  0.145727  0.432576  0.012062  
            Cellina-base-random     4.126696  0.169116  0.469366  0.029045  
            SpatialProp             7.440094  0.035459  0.641832  0.040702  
            SpatialProp-random      7.483979  0.019528  0.662955  0.037157  
            mean                    0.132188  0.009945  0.461127  0.023426  
            terra-lora-ep5          1.709620  0.106576  0.464786  0.030393  
            terra-lora-ep5-random   1.715885  0.122621  0.477342  0.029262  
full        Cellina-GAT             4.014472  0.217723  0.530210  0.057388  
            Cellina-GAT-random      4.061126  0.250042  0.612370  0.041242  
            Cellina-base            4.134229  0.147872  0.496071  0.018788  
            Cellina-base-random     4.143967  0.173536  0.587756  0.030776  
            SpatialProp             7.454111  0.024091  0.682143  0.085501  
            SpatialProp-random      7.469573  0.017814  0.703670  0.087957  
            mean                    0.136562  0.013408  0.536585  0.029357  
            terra-lora-ep5          1.719037  0.122693  0.575974  0.036764  
            terra-lora-ep5-random   1.716829  0.119086  0.592942  0.043974

In [6]:
n_seeds = per_seed.groupby(["type", "baseline"])["seed"].nunique().rename("n_seeds")

summary_fmt = per_seed.groupby(["type", "baseline"])[METRICS].agg(
    lambda x: f"{x.mean():.2f} \u00b1 {x.std():.2f}"
)
summary_fmt = summary_fmt.join(n_seeds)
summary_fmt

Pearson    Precision   E-distance  \
type        baseline                                                        
KO-specific Cellina-GAT             0.36 ± 0.18  0.64 ± 0.05  4.00 ± 0.26   
            Cellina-GAT-random      0.08 ± 0.07  0.52 ± 0.03  4.06 ± 0.23   
            Cellina-base            0.33 ± 0.08  0.68 ± 0.03  4.16 ± 0.15   
            Cellina-base-random    -0.00 ± 0.05  0.48 ± 0.04  4.13 ± 0.17   
            SpatialProp             0.16 ± 0.06  0.55 ± 0.06  7.44 ± 0.04   
            SpatialProp-random      0.11 ± 0.05  0.52 ± 0.06  7.48 ± 0.02   
            mean                    0.06 ± 0.06  0.51 ± 0.03  0.13 ± 0.01   
            terra-lora-ep5          0.03 ± 0.07  0.53 ± 0.05  1.71 ± 0.11   
            terra-lora-ep5-random  -0.03 ± 0.06  0.50 ± 0.04  1.72 ± 0.12   
full        Cellina-GAT             0.40 ± 0.22  0.67 ± 0.12  4.01 ± 0.22   
            Cellina-GAT-random     -0.01 ± 0.17  0.50 ± 0.12  4.06 ± 0.25   
            Cellina-base            0.53 ± 0.05  0.76 ± 0.03  4.13 ± 0.15   
            Cellina-base-random     0.11 ± 0.23  0.55 ± 0.06  4.14 ± 0.17   
            SpatialProp             0.26 ± 0.16  0.61 ± 0.08  7.45 ± 0.02   
            SpatialProp-random      0.19 ± 0.17  0.58 ± 0.09  7.47 ± 0.02   
            mean                    0.51 ± 0.13  0.73 ± 0.08  0.14 ± 0.01   
            terra-lora-ep5          0.15 ± 0.12  0.55 ± 0.05  1.72 ± 0.12   
            terra-lora-ep5-random   0.09 ± 0.14  0.53 ± 0.06  1.72 ± 0.12   

                                      RMSE_LFC  n_seeds  
type        baseline                                     
KO-specific Cellina-GAT            0.42 ± 0.04        5  
            Cellina-GAT-random     0.47 ± 0.03        5  
            Cellina-base           0.43 ± 0.01        5  
            Cellina-base-random    0.47 ± 0.03        5  
            SpatialProp            0.64 ± 0.04        5  
            SpatialProp-random     0.66 ± 0.04        5  
            mean                   0.46 ± 0.02        5  
            terra-lora-ep5         0.46 ± 0.03        5  
            terra-lora-ep5-random  0.48 ± 0.03        5  
full        Cellina-GAT            0.53 ± 0.06        5  
            Cellina-GAT-random     0.61 ± 0.04        5  
            Cellina-base           0.50 ± 0.02        5  
            Cellina-base-random    0.59 ± 0.03        5  
            SpatialProp            0.68 ± 0.09        5  
            SpatialProp-random     0.70 ± 0.09        5  
            mean                   0.54 ± 0.03        5  
            terra-lora-ep5         0.58 ± 0.04        5  
            terra-lora-ep5-random  0.59 ± 0.04        5

## LaTeX table

Writes the manuscript-format table (`type` × `method` rows, Pearson/Precision/E-distance/
RMSE_LFC columns, mean ± std over seeds, best value per column **within each type section**
bolded) to `results/invivo_summary_table.tex`. Row order and which baseline maps to which
display name are fixed below; a baseline not in that mapping (e.g. a new model) still prints,
under its raw name, appended after the known rows -- so nothing silently goes missing.

In [7]:
TEX_PATH = os.path.join(RESULTS_DIR, "invivo_summary_table.tex")

METRIC_BETTER = {"Pearson": "max", "Precision": "max", "E-distance": "min", "RMSE_LFC": "min"}

LATEX_HEADER = (r"Type & Method & Pearson $\uparrow$ & Precision$_{\mathrm{signed}}$ $\uparrow$ "
                 r"& E-distance $\downarrow$ & RMSE\textsubscript{LFC} $\downarrow$ \\")

TYPE_SECTIONS = [("KO-specific", "KO-specific"), ("full", "Full")]

# baseline (as written by scripts/invivo_multi_seed.py) -> display name in the table
FIXED_METHOD_NAMES = {
    "Cellina-base": "Cellina",
    "Cellina-GAT": "Cellina-GAT",
    "SpatialProp": "SpatialProp",
    "mean": "Mean shift",
    "Cellina-base-random": "Cellina-rand",
    "Cellina-GAT-random": "Cellina-GAT-rand",
    "SpatialProp-random": "SpatialProp-rand",
}
# fixed row order (only rows actually present in the data are emitted, so the \multirow count
# below is never stale even as models get added/removed)
LATEX_METHOD_ORDER = ["Cellina", "Cellina-GAT", "SpatialProp", "Mean shift",
                       "Cellina-rand", "Cellina-GAT-rand", "SpatialProp-rand",
                       "TERRA-LoRA", "TERRA-LoRA-rand"]


def method_name(baseline):
    if baseline in FIXED_METHOD_NAMES:
        return FIXED_METHOD_NAMES[baseline]
    if baseline.startswith("terra-lora-ep"):  # embeds the LoRA epoch, e.g. terra-lora-ep5(-random)
        return "TERRA-LoRA-rand" if baseline.endswith("-random") else "TERRA-LoRA"
    return baseline  # unknown baseline: still shown, just unmapped and sorted to the end


def make_latex_table(summary, metrics=METRICS):
    tbl = summary.reset_index()
    baseline_col = tbl[("baseline", "")] if ("baseline", "") in tbl.columns else tbl["baseline"]
    type_col = tbl[("type", "")] if ("type", "") in tbl.columns else tbl["type"]
    tbl["Method"] = [method_name(b) for b in baseline_col]
    tbl["_type"] = type_col.values

    lines = [r"\begin{tabular}{llcccc}", r"\toprule", LATEX_HEADER, r"\midrule"]
    for type_val, type_label in TYPE_SECTIONS:
        sub = tbl[tbl["_type"] == type_val].set_index("Method")
        row_order = [m for m in LATEX_METHOD_ORDER if m in sub.index]
        row_order += [m for m in sub.index if m not in LATEX_METHOD_ORDER]  # unmapped, appended
        sub = sub.loc[row_order]
        if sub.empty:
            continue

        best = {m: (sub[(m, "mean")].idxmax() if METRIC_BETTER[m] == "max" else sub[(m, "mean")].idxmin())
                for m in metrics}

        n = len(sub)
        lines.append(r"\multirow{%d}{*}{\shortstack{KO-\\specific}}" % n
                     if type_val == "KO-specific" else type_label)
        for method, row in sub.iterrows():
            cells = []
            for metric in metrics:
                s = f"{row[(metric, 'mean')]:.2f} $\\pm$ {row[(metric, 'std')]:.2f}"
                if best[metric] == method:
                    s = r"\textbf{%s}" % s
                cells.append(s)
            lines.append("& " + method + " & " + " & ".join(cells) + r" \\")
        lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"  # overwrite the last section's trailing \midrule
    lines.append(r"\end{tabular}")
    return "\n".join(lines)


tex = make_latex_table(summary)
with open(TEX_PATH, "w") as f:
    f.write(tex + "\n")
print(f"wrote {TEX_PATH}\n")
print(tex)

wrote ../../results/invivo_summary_table.tex

\begin{tabular}{llcccc}
\toprule
Type & Method & Pearson $\uparrow$ & Precision$_{\mathrm{signed}}$ $\uparrow$ & E-distance $\downarrow$ & RMSE\textsubscript{LFC} $\downarrow$ \\
\midrule
\multirow{9}{*}{\shortstack{KO-\\specific}}
& Cellina & 0.33 $\pm$ 0.08 & \textbf{0.68 $\pm$ 0.03} & 4.16 $\pm$ 0.15 & 0.43 $\pm$ 0.01 \\
& Cellina-GAT & \textbf{0.36 $\pm$ 0.18} & 0.64 $\pm$ 0.05 & 4.00 $\pm$ 0.26 & \textbf{0.42 $\pm$ 0.04} \\
& SpatialProp & 0.16 $\pm$ 0.06 & 0.55 $\pm$ 0.06 & 7.44 $\pm$ 0.04 & 0.64 $\pm$ 0.04 \\
& Mean shift & 0.06 $\pm$ 0.06 & 0.51 $\pm$ 0.03 & \textbf{0.13 $\pm$ 0.01} & 0.46 $\pm$ 0.02 \\
& Cellina-rand & -0.00 $\pm$ 0.05 & 0.48 $\pm$ 0.04 & 4.13 $\pm$ 0.17 & 0.47 $\pm$ 0.03 \\
& Cellina-GAT-rand & 0.08 $\pm$ 0.07 & 0.52 $\pm$ 0.03 & 4.06 $\pm$ 0.23 & 0.47 $\pm$ 0.03 \\
& SpatialProp-rand & 0.11 $\pm$ 0.05 & 0.52 $\pm$ 0.06 & 7.48 $\pm$ 0.02 & 0.66 $\pm$ 0.04 \\
& TERRA-LoRA & 0.03 $\pm$ 0.07 & 0.53 $\pm$ 0.05 & 1.71 